# Single Cell data
CC 2026-08-11

## 1. Setup

In [1]:
import retinanalysis as ra
import matplotlib.pyplot as plt
import numpy as np

# Read-only single-cell database queries and notebook browsers.
from retinanalysis.SCutils import explore as sc
# Map new h5 files when needed.
report = ra.SCutils.update_single_cell_json()

Single-cell drive: /Volumes/ChrisNewSSD
H5 folder: /Volumes/ChrisNewSSD/single_cell/chris_data/h5
JSON folder: /Volumes/ChrisNewSSD/single_cell/chris_data/json
No new H5 files; every H5 already has matching JSON.


## 2. Populate and refresh the database

`populate_database()` ingests new experiments, refreshes experiments whose JSON changed, and returns the database freshness check in the same report. Canonical experiment names such as `YYYY-MM-DD_X.h5` are included; auxiliary and legacy files are ignored. By default only metadata and tags JSON files trigger a refresh. Pass `watch_data_file=True` to include H5 modification times.

In [ ]:
# One call handles ingest, refresh, and the post-ingest stale-file check.
summary = ra.populate_database()
df_db = summary['experiments']
df_stale = summary['stale']

print(f"newly added : {len(summary['added'])}")
print(f"refreshed   : {len(summary['updated'])}")
print(f"errored     : {len(summary['skipped'])}")
print(f"database    : {len(df_db)} experiments; {len(df_stale)} still out of date")

if len(df_stale):
    display(df_stale[['exp_name', 'date_added', 'source_mtime', 'source_file']])

# ra.purge_experiments('2026-06-04_G')
# ra.purge_experiments(['2026-05-06_E', '2026-05-08_E'])

# Drop only rows that remain stale after populate.
# ra.purge_experiments(df_stale['exp_name'].tolist())

# Wipe the whole database (requires the literal confirmation token).
# ra.purge_database(confirm='YES_DELETE_ALL')

print(f'{len(df_db)} experiments currently in the database.')

## 3. List single-cell experiments

Experiments are shown in separate `chris_data` and `fred_data` tables. Each protocol gets its own row so a date is easier to scan; repeated experiment, project, and short cell-type values are visually grouped. Owner and species are kept only for the cascading browser and are omitted from the table.

In [ ]:
# Section 3 has exactly these visible columns. Owner and species remain
# internal to the cascading browser and are not included here.
df_sc_exps = sc.list_experiments(show=False)

# Accept both the current singular column and an older, comma-joined
# `protocols` column from a module already cached in this kernel.
if 'protocols' in df_sc_exps.columns:
    df_sc_exps['protocol'] = df_sc_exps.pop('protocols').fillna('?').str.split(r',\s*', regex=True)
    df_sc_exps = df_sc_exps.explode('protocol', ignore_index=True)

section3_columns = ['exp_name', 'cell_types', 'protocol']
df_sc_exps = (df_sc_exps.loc[:, section3_columns]
              .drop_duplicates()
              .sort_values(['exp_name', 'protocol'], ignore_index=True))
sc.tree_table(df_sc_exps, levels=['exp_name', 'project', 'cell_types'], height=500)

### 4.1 Protocol coverage by species

One row per short protocol. Counts are unique experiment dates, not epoch blocks: `primate_dates` and `mouse_dates` show species-specific coverage, and `total_dates` includes every species.


In [ ]:
# Scrollable protocol inventory, sorted by the number of dates.
df_protocol_inventory = sc.protocol_inventory(height=500)


## 4. Find experiments by protocol

Search protocol names case-insensitively. The returned DataFrame remains one row per epoch block. The expanded block table shows fixed NDF settings plus the actual numeric filter-wheel reading for every block; an embedded `FWx` label is not used in place of the wheel reading.

In [2]:
df_blocks = sc.find_blocks('LinearEquivalentDiscConeLin')

193 blocks | 16 experiments | 1 protocol(s) matching 'LinearEquivalentDiscConeLin'


exp_name,blocks,protocols,block_ids
2026-04-10_G,3,LinearEquivalentDiscConeLin,"37324, 37332-37333"
2026-04-15_E,16,LinearEquivalentDiscConeLin,"34057-34069, 34108-34110"
2026-04-17_E,10,LinearEquivalentDiscConeLin,34210-34219
2026-04-24_E,6,LinearEquivalentDiscConeLin,34368-34373
2026-04-28_E,29,LinearEquivalentDiscConeLin,"33062-33072, 33074, 33076-33077, 33079-33081, 33084, 33087, 33092-33097, 33099, 33102, 33105-33106"
2026-05-06_E,13,LinearEquivalentDiscConeLin,"36418-36426, 36429-36432"
2026-05-06_G,3,LinearEquivalentDiscConeLin,37001-37003
2026-05-07_G,1,LinearEquivalentDiscConeLin,37121
2026-05-08_E,10,LinearEquivalentDiscConeLin,"36461, 36559-36561, 36564-36569"
2026-05-08_G,11,LinearEquivalentDiscConeLin,"37123-37125, 37129-37132, 37138-37139, 37149-37150"


exp_name,protocol,block_id,NDF + FW
2026-04-10_G,LinearEquivalentDiscConeLin,37324,EL2 + FW0.5
2026-04-10_G,LinearEquivalentDiscConeLin,37332,EL2 + FW0.5
2026-04-10_G,LinearEquivalentDiscConeLin,37333,EL2 + FW0.5
2026-04-15_E,LinearEquivalentDiscConeLin,34057,EL3 + FW0
2026-04-15_E,LinearEquivalentDiscConeLin,34058,EL3 + FW0
2026-04-15_E,LinearEquivalentDiscConeLin,34059,EL3 + FW1
2026-04-15_E,LinearEquivalentDiscConeLin,34060,EL3 + FW1
2026-04-15_E,LinearEquivalentDiscConeLin,34061,EL3 + FW1
2026-04-15_E,LinearEquivalentDiscConeLin,34062,EL3 + FW1
2026-04-15_E,LinearEquivalentDiscConeLin,34063,EL3 + FW0


## 5. Browse and summarize experiments

Use the cascading menus to select data owner, species, and experiment. Each experiment date appears once in the menu, regardless of how many protocols ran that day. The overview is organized as cell → epoch group (group label) → protocol, with block and epoch counts. Then select an epoch block and click **Load original traces** to read and display every unprocessed Amp1 epoch trace from the H5 file.

In [3]:
# Pass one row per date to the browser; Section 3 intentionally has one
# row per protocol and must not duplicate dates in this menu.
browser_dates = df_sc_exps[['exp_name']].drop_duplicates(ignore_index=True)
experiment_browser = sc.summarize_experiments(browser_dates)

NameError: name 'df_sc_exps' is not defined